In [ ]:
!pip install -q geopandas requests ipywidgets shapely fiona pyproj rtree ipyfilechooser folium

In [ ]:
import os
import json
import time
import zipfile
import requests
import geopandas as gpd
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from typing import Optional, List, Dict, Tuple
from dataclasses import dataclass, field
from io import BytesIO, StringIO
import uuid
import ipywidgets as widgets
from IPython.display import display, clear_output
from shapely.geometry import box
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    IN_COLAB = False

In [ ]:
@dataclass
class ComparisonConfig:
    include_previous_months: bool = True
    previous_months_count: int = 2
    include_previous_year: bool = True
    previous_years_count: int = 1
    include_next_month: bool = True
    
    def generate_periods(self, start_date: datetime, end_date: datetime) -> List[Dict]:
        periods = []
        
        if self.include_previous_months:
            for i in range(1, self.previous_months_count + 1):
                periods.append({
                    'type': 'previous_month',
                    'offset': i,
                    'start': start_date - relativedelta(months=i),
                    'end': end_date - relativedelta(months=i),
                    'description': f'{i} міс. тому',
                    'folder_suffix': f'prev_{i}_months'
                })
        
        if self.include_previous_year:
            for i in range(1, self.previous_years_count + 1):
                periods.append({
                    'type': 'previous_year',
                    'offset': i,
                    'start': start_date - relativedelta(years=i),
                    'end': end_date - relativedelta(years=i),
                    'description': f'Ті ж місяці {i} р. тому',
                    'folder_suffix': f'prev_{i}_year'
                })
        
        if self.include_next_month:
            periods.append({
                'type': 'next_month',
                'offset': 1,
                'start': start_date + relativedelta(months=1),
                'end': end_date + relativedelta(months=1),
                'description': 'Наступний місяць',
                'folder_suffix': 'next_1_month'
            })
        
        return periods


@dataclass
class SessionLog:
    session_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    started_at: datetime = field(default_factory=datetime.now)
    region_name: str = ""
    download_mode: str = ""
    output_folder: str = ""
    fire_period: Optional[Tuple[datetime, datetime]] = None
    comparison_config: Optional[ComparisonConfig] = None
    aoi_file_name: Optional[str] = None
    downloaded_files: List[Dict] = field(default_factory=list)
    firms_data_file: Optional[str] = None
    boundary_file: Optional[str] = None
    errors: List[str] = field(default_factory=list)
    data_sources: List[str] = field(default_factory=list)
    
    def add_file(self, filename: str, file_type: str, size_bytes: int = 0):
        self.downloaded_files.append({'filename': filename, 'type': file_type, 'size_bytes': size_bytes, 'downloaded_at': datetime.now().isoformat()})
    
    def add_error(self, error: str):
        self.errors.append(f"[{datetime.now().strftime('%H:%M:%S')}] {error}")
    
    def save(self, folder: str):
        log_dict = {
            'session_id': self.session_id,
            'started_at': self.started_at.isoformat(),
            'ended_at': datetime.now().isoformat(),
            'region_name': self.region_name,
            'download_mode': self.download_mode,
            'output_folder': self.output_folder,
            'downloaded_files': self.downloaded_files,
            'errors': self.errors,
            'data_sources': self.data_sources
        }
        with open(os.path.join(folder, 'download_log.json'), 'w', encoding='utf-8') as f:
            json.dump(log_dict, f, indent=2, ensure_ascii=False)

In [ ]:
class GeoBoundariesClient:
    BOUNDARIES_URLS = [
        "https://raw.githubusercontent.com/slawomirmatuszak/ukrainian_geodata/main/regiony.geojson",
        "https://raw.githubusercontent.com/EugeneBorshch/ukraine_geojson/master/ukraine_regions.geojson",
    ]
    
    UA_NAMES = {
        'vinnytsia': 'Вінницька область', 'volyn': 'Волинська область',
        'dnipropetrovsk': 'Дніпропетровська область', 'donetsk': 'Донецька область',
        'zhytomyr': 'Житомирська область', 'zakarpattia': 'Закарпатська область',
        'transcarpathia': 'Закарпатська область', 'zaporizhzhia': 'Запорізька область',
        'ivano-frankivsk': 'Івано-Франківська область', 'kyiv': 'Київська область',
        'kiev': 'Київська область', 'kirovohrad': 'Кіровоградська область',
        'luhansk': 'Луганська область', 'lviv': 'Львівська область',
        'mykolaiv': 'Миколаївська область', 'odesa': 'Одеська область',
        'odessa': 'Одеська область', 'poltava': 'Полтавська область',
        'rivne': 'Рівненська область', 'sumy': 'Сумська область',
        'ternopil': 'Тернопільська область', 'kharkiv': 'Харківська область',
        'kherson': 'Херсонська область', 'khmelnytskyi': 'Хмельницька область',
        'cherkasy': 'Черкаська область', 'chernivtsi': 'Чернівецька область',
        'chernihiv': 'Чернігівська область', 'crimea': 'АР Крим',
        'sevastopol': 'м. Севастополь', 'kyiv city': 'м. Київ',
    }
    
    @classmethod
    def get_ukraine_oblasts(cls) -> gpd.GeoDataFrame:
        try:
            api_url = "https://www.geoboundaries.org/api/current/gbOpen/UKR/ADM1/"
            response = requests.get(api_url, timeout=30)
            response.raise_for_status()
            api_data = response.json()
            geojson_url = api_data.get('gjDownloadURL') or api_data.get('simplifiedGeometryGeoJSON')
            if geojson_url:
                gdf = gpd.read_file(geojson_url).set_crs(epsg=4326, allow_override=True)
                print(f"✅ Завантажено {len(gdf)} областей з geoBoundaries")
                return cls._process_oblasts(gdf)
        except Exception as e:
            print(f"⚠️ geoBoundaries недоступний: {e}")
        
        for url in cls.BOUNDARIES_URLS:
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                if 'text/html' in response.headers.get('content-type', ''):
                    continue
                data = response.json()
                if 'features' in data:
                    gdf = gpd.GeoDataFrame.from_features(data['features']).set_crs(epsg=4326, allow_override=True)
                    print(f"✅ Завантажено {len(gdf)} областей")
                    return cls._process_oblasts(gdf)
            except:
                continue
        raise Exception("Не вдалося завантажити межі областей!")
    
    @classmethod
    def _process_oblasts(cls, gdf):
        name_col = next((col for col in ['shapeName', 'name', 'NAME_1', 'ADM1_EN', 'region'] if col in gdf.columns), gdf.columns[0])
        
        def get_ua_name(en_name):
            if pd.isna(en_name): return 'Невідома область'
            name_lower = str(en_name).lower().strip()
            for key, ua_name in cls.UA_NAMES.items():
                if key in name_lower or name_lower in key:
                    return ua_name
            return f"{en_name} область" if not any(c in str(en_name) for c in 'абвгдеєжзиіїйклмнопрстуфхцчшщьюя') else en_name
        
        gdf['name_en'] = gdf[name_col].astype(str)
        gdf['name_ua'] = gdf['name_en'].apply(get_ua_name)
        gdf['display_name'] = gdf['name_ua']
        return gdf.sort_values('name_ua').reset_index(drop=True)


class FIRMSClient:
    AREA_API_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
    NRT_DAYS_LIMIT = 60
    SOURCES_FOR_UI = [
        ('MODIS (рекомендовано для України)', 'MODIS'),
        ('VIIRS S-NPP (375м)', 'VIIRS_SNPP'),
        ('VIIRS NOAA-20 (375м)', 'VIIRS_NOAA20'),
        ('VIIRS NOAA-21 (375м)', 'VIIRS_NOAA21'),
    ]
    
    def __init__(self, api_key: str):
        self.api_key = api_key
    
    def get_fires(self, bbox, start_date: datetime, end_date: datetime, source: str = 'MODIS') -> gpd.GeoDataFrame:
        area = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
        today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        all_fires, current_date = [], start_date
        
        print(f"   📡 Джерело: {source} | 📅 {start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')} | Днів: {(end_date - start_date).days + 1}")
        
        while current_date <= end_date:
            chunk_end = min(current_date + timedelta(days=9), end_date)
            chunk_days = (chunk_end - current_date).days + 1
            full_source = f"{source}_NRT" if (today - current_date).days < self.NRT_DAYS_LIMIT else f"{source}_SP"
            url = f"{self.AREA_API_URL}/{self.api_key}/{full_source}/{area}/{chunk_days}/{current_date.strftime('%Y-%m-%d')}"
            
            print(f"   📥 {current_date.strftime('%Y-%m-%d')} → {chunk_end.strftime('%Y-%m-%d')}...", end=" ")
            try:
                response = requests.get(url, timeout=60)
                if response.status_code == 200:
                    text = response.text.strip()
                    if text and 'latitude' in text and not text.startswith('<'):
                        df = pd.read_csv(StringIO(text))
                        if not df.empty and 'latitude' in df.columns:
                            all_fires.append(df)
                            print(f"✓ {len(df)}")
                        else: print("-")
                    else: print("-")
                else: print(f"⚠️ {response.status_code}")
            except Exception as e: print(f"⚠️ {str(e)[:25]}")
            current_date = chunk_end + timedelta(days=1)
        
        if not all_fires:
            print("   ⚠️ Пожеж не знайдено")
            return gpd.GeoDataFrame()
        
        combined_df = pd.concat(all_fires, ignore_index=True)
        if 'acq_date' in combined_df.columns:
            combined_df = combined_df.drop_duplicates(subset=['latitude', 'longitude', 'acq_date', 'acq_time'], keep='first')
            combined_df['acq_date'] = pd.to_datetime(combined_df['acq_date'])
            combined_df = combined_df[(combined_df['acq_date'] >= start_date.strftime('%Y-%m-%d')) & (combined_df['acq_date'] <= end_date.strftime('%Y-%m-%d'))]
        
        if combined_df.empty: return gpd.GeoDataFrame()
        gdf = gpd.GeoDataFrame(combined_df, geometry=gpd.points_from_xy(combined_df.longitude, combined_df.latitude), crs="EPSG:4326")
        print(f"   ✅ Всього: {len(gdf)} точок")
        return gdf
    
    @staticmethod
    def cluster_fires(fires_gdf: gpd.GeoDataFrame, buffer_km: float = 2.0) -> List[Dict]:
        if fires_gdf.empty: return []
        fires_proj = fires_gdf.to_crs(epsg=3857)
        dissolved = fires_proj.buffer(buffer_km * 1000).unary_union
        geoms = [dissolved] if dissolved.geom_type == 'Polygon' else list(dissolved.geoms) if dissolved.geom_type == 'MultiPolygon' else []
        
        clusters = []
        for i, geom in enumerate(geoms):
            cluster_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:3857").to_crs("EPSG:4326")
            cluster_geom = cluster_gdf.iloc[0].geometry
            clusters.append({'id': f'cluster_{i+1}', 'bbox': tuple(cluster_gdf.total_bounds), 'geometry': cluster_geom, 'n_hotspots': len(fires_gdf[fires_gdf.within(cluster_geom)])})
        return clusters


class PlanetClient:
    BASE_URL = "https://api.planet.com/basemaps/v1"
    MONTHS_EN = {'01': 'january', '02': 'february', '03': 'march', '04': 'april', '05': 'may', '06': 'june', '07': 'july', '08': 'august', '09': 'september', '10': 'october', '11': 'november', '12': 'december'}
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.auth = requests.auth.HTTPBasicAuth(api_key, "")
        self._mosaics_cache = None
    
    def test_connection(self) -> bool:
        try:
            requests.get(f"{self.BASE_URL}/mosaics", auth=self.auth, params={"_limit": 1}, timeout=30).raise_for_status()
            print("   ✅ Planet API підключено")
            return True
        except Exception as e:
            print(f"   ❌ Planet: {e}")
            return False
    
    def get_mosaic_list(self) -> Dict[str, str]:
        if self._mosaics_cache: return self._mosaics_cache
        all_mosaics = {}
        for pattern in ["global_monthly", "planet_medres", "2025", "2024", "2023", "2022", "2021", "2020"]:
            try:
                r = requests.get(f"{self.BASE_URL}/mosaics", auth=self.auth, params={"name__contains": pattern, "_limit": 300}, timeout=60)
                r.raise_for_status()
                for m in r.json().get("mosaics", []):
                    if m["name"] not in all_mosaics: all_mosaics[m["name"]] = m["id"]
            except: continue
        self._mosaics_cache = all_mosaics
        return all_mosaics
    
    def find_mosaic(self, year: str, month: str) -> Optional[Dict]:
        mosaics = self.get_mosaic_list()
        month_name = self.MONTHS_EN.get(month, '')
        for pattern in [f"planet_medres_visual_global_monthly_{year}_{month_name}", f"global_monthly_{year}_{month_name}", f"global_monthly_{year}_{month}"]:
            for name, mosaic_id in mosaics.items():
                if pattern.lower() in name.lower(): return {'name': name, 'id': mosaic_id}
        return None
    
    def get_quads_for_aoi(self, mosaic_id: str, aoi_gdf: gpd.GeoDataFrame) -> List[Dict]:
        bounds = aoi_gdf.total_bounds
        params = {"bbox": ",".join(str(v) for v in bounds), "_page_size": 500}
        response = requests.get(f"{self.BASE_URL}/mosaics/{mosaic_id}/quads", auth=self.auth, params=params, timeout=60)
        response.raise_for_status()
        quads_response = response.json()
        items = quads_response.get("items", [])
        while quads_response["_links"].get("_next"):
            response = requests.get(quads_response["_links"]["_next"], auth=self.auth, timeout=60)
            response.raise_for_status()
            quads_response = response.json()
            items.extend(quads_response.get("items", []))
        if not items: return []
        quads_gdf = gpd.GeoDataFrame([{"quad_id": q["id"], "geometry": box(*q["bbox"])} for q in items]).set_crs(epsg=4326)
        filtered_ids = set(gpd.sjoin(quads_gdf, aoi_gdf[['geometry']], how="inner", predicate="intersects")['quad_id'].values)
        return [q for q in items if q["id"] in filtered_ids]
    
    def download_tiles(self, quads: List[Dict], output_dir: str) -> Tuple[int, int, int]:
        existing = {f.replace('.tif', '') for f in os.listdir(output_dir) if f.endswith('.tif')}
        to_download = [q for q in quads if q["id"] not in existing]
        if not to_download: return len(existing), 0, 0
        
        session = requests.Session()
        session.headers.update({'User-Agent': 'Mozilla/5.0'})
        downloaded, failed = 0, 0
        
        for quad in to_download:
            output_file = os.path.join(output_dir, f"{quad['id']}.tif")
            download_url = quad["_links"]["download"]
            if "api_key=" not in download_url:
                download_url = f"{download_url}{'&' if '?' in download_url else '?'}api_key={self.api_key}"
            
            for attempt in range(3):
                try:
                    response = session.get(download_url, allow_redirects=True, timeout=180)
                    response.raise_for_status()
                    if len(response.content) > 1000:
                        with open(output_file, 'wb') as f: f.write(response.content)
                        downloaded += 1
                        break
                except: time.sleep(3) if attempt < 2 else None
            else: failed += 1
            time.sleep(0.3)
        session.close()
        return len(existing), downloaded, failed

In [ ]:
def parse_uploaded_file(file_path: str) -> gpd.GeoDataFrame:
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ['.geojson', '.json', '.shp']: return gpd.read_file(file_path)
    if ext == '.kml':
        gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
        return gpd.read_file(file_path, driver='KML')
    if ext == '.kmz':
        with zipfile.ZipFile(file_path, 'r') as z:
            for name in z.namelist():
                if name.endswith('.kml'):
                    gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
                    return gpd.read_file(BytesIO(z.read(name)), driver='KML')
    raise ValueError(f"Непідтримуваний формат: {ext}")


def create_folder_structure(base_path: str, region_name: str, mode: str, fire_period=None, comparison_config=None) -> Dict[str, str]:
    safe_name = region_name.replace(' ', '_').replace('/', '_').replace('\\', '_')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    folder_name = f"{safe_name}_{mode}_{fire_period[0].strftime('%Y%m%d')}_to_{fire_period[1].strftime('%Y%m%d')}_{timestamp}" if fire_period else f"{safe_name}_{mode}_{timestamp}"
    
    session_folder = os.path.join(base_path, folder_name)
    folders = {'session': session_folder, 'data': os.path.join(session_folder, 'data'), 'logs': os.path.join(session_folder, 'logs')}
    for folder in folders.values(): os.makedirs(folder, exist_ok=True)
    
    if mode == 'fire_areas' and fire_period and comparison_config:
        folders['fire_period'] = os.path.join(session_folder, f"fire_period_{fire_period[0].strftime('%Y%m%d')}_to_{fire_period[1].strftime('%Y%m%d')}")
        os.makedirs(folders['fire_period'], exist_ok=True)
        for period in comparison_config.generate_periods(fire_period[0], fire_period[1]):
            period_folder = os.path.join(session_folder, f"comparison_{period['folder_suffix']}_{period['start'].strftime('%Y%m%d')}_to_{period['end'].strftime('%Y%m%d')}")
            os.makedirs(period_folder, exist_ok=True)
            folders[f"comparison_{period['folder_suffix']}"] = period_folder
    return folders


def save_geodataframe(gdf: gpd.GeoDataFrame, folder: str, filename: str) -> str:
    filepath = os.path.join(folder, filename)
    gdf.to_file(filepath, driver='GeoJSON')
    return filepath

In [ ]:
class ForestFireUI:
    def __init__(self):
        self.oblasts_gdf = None
        self.aoi_gdf = None
        self.log = None
        self._create_widgets()
        self._load_oblasts()
    
    def _create_widgets(self):
        style = {'description_width': '150px'}
        layout = widgets.Layout(width='500px')
        
        self.oblast_dropdown = widgets.Dropdown(options=[('Завантаження...', None)], description='Область:', style=style, layout=layout)
        self.mode_radio = widgets.RadioButtons(
            options=[('Вся область', 'full_oblast'), ('Зона інтересу', 'aoi'), ('Лише пожежі', 'fire_areas')],
            value='fire_areas', description='Режим:', style=style, layout=widgets.Layout(width='400px')
        )
        self.mode_radio.observe(self._on_mode_change, names='value')
        
        today = datetime.now()
        self.fire_start_date = widgets.DatePicker(description='Початок:', value=(today - timedelta(days=30)).date(), style=style)
        self.fire_end_date = widgets.DatePicker(description='Кінець:', value=today.date(), style=style)
        self.firms_source = widgets.Dropdown(options=FIRMSClient.SOURCES_FOR_UI, value='MODIS', description='Джерело FIRMS:', style=style, layout=layout)
        
        self.include_prev_months = widgets.Checkbox(value=True, description='Попередні місяці', style=style)
        self.prev_months_count = widgets.IntSlider(value=2, min=1, max=6, description='Кількість:', style=style)
        self.include_prev_year = widgets.Checkbox(value=True, description='Попередній рік', style=style)
        self.prev_years_count = widgets.IntSlider(value=1, min=1, max=3, description='Кількість:', style=style)
        self.include_next_month = widgets.Checkbox(value=True, description='Наступний місяць (після пожежі)', style=style)
        
        self.fire_options_box = widgets.VBox([
            widgets.HTML('<b>Період пожежі:</b>'),
            widgets.HBox([self.fire_start_date, self.fire_end_date]),
            self.firms_source,
            widgets.HTML('<b>Порівняльні знімки:</b>'),
            widgets.HBox([self.include_prev_months, self.prev_months_count]),
            widgets.HBox([self.include_prev_year, self.prev_years_count]),
            self.include_next_month
        ])
        
        self.file_upload = widgets.FileUpload(accept='.geojson,.json,.shp,.kml,.kmz,.zip', multiple=False, description='Файл')
        self.file_upload.observe(self._on_file_upload, names='value')
        self.aoi_status = widgets.HTML('<i>Файл не завантажено</i>')
        self.aoi_options_box = widgets.VBox([widgets.HTML('<b>Завантажте файл з межами:</b>'), self.file_upload, self.aoi_status])
        self.aoi_options_box.layout.display = 'none'
        
        current_year = datetime.now().year
        self.years_select = widgets.SelectMultiple(options=[str(y) for y in range(current_year, 2018, -1)], value=[str(current_year)], description='Роки:', style=style, rows=5)
        months = [(f'{i:02d}', f'{i:02d}') for i in range(1, 13)]
        self.months_select = widgets.SelectMultiple(options=months, value=[f"{datetime.now().month:02d}"], description='Місяці:', style=style, rows=6)
        self.full_oblast_options_box = widgets.VBox([widgets.HTML('<b>Періоди:</b>'), widgets.HBox([self.years_select, self.months_select])])
        self.full_oblast_options_box.layout.display = 'none'
        
        self.planet_key = widgets.Password(description='Planet API Key:', style=style, layout=layout)
        self.firms_key = widgets.Password(description='FIRMS MAP Key:', style=style, layout=layout)
        self.output_folder = widgets.Text(description='Папка:', value='/content/drive/MyDrive/ForestFireData' if IN_COLAB else './output', style=style, layout=layout)
        
        self.check_btn = widgets.Button(description='Перевірити', button_style='info', layout=widgets.Layout(width='150px'))
        self.check_btn.on_click(self._on_check_click)
        self.start_button = widgets.Button(description='Завантажити', button_style='success', layout=widgets.Layout(width='150px'))
        self.start_button.on_click(self._on_start_click)
        
        self.progress = widgets.IntProgress(value=0, min=0, max=100, description='', layout=widgets.Layout(width='400px'))
        self.progress.layout.display = 'none'
        self.status_output = widgets.Output()
    
    def _load_oblasts(self):
        try:
            self.oblasts_gdf = GeoBoundariesClient.get_ukraine_oblasts()
            self.oblast_dropdown.options = [('-- Оберіть область --', None)] + [(row['display_name'], idx) for idx, row in self.oblasts_gdf.iterrows()]
        except Exception as e:
            self.oblast_dropdown.options = [(f'Помилка: {str(e)[:50]}', None)]
    
    def _on_mode_change(self, change):
        mode = change['new']
        self.fire_options_box.layout.display = 'block' if mode == 'fire_areas' else 'none'
        self.aoi_options_box.layout.display = 'block' if mode == 'aoi' else 'none'
        self.full_oblast_options_box.layout.display = 'block' if mode == 'full_oblast' else 'none'
    
    def _on_file_upload(self, change):
        if change['new']:
            try:
                uploaded = list(change['new'].values())[0]
                filename = list(change['new'].keys())[0]
                temp_path = f'/tmp/{filename}'
                with open(temp_path, 'wb') as f: f.write(uploaded['content'])
                self.aoi_gdf = parse_uploaded_file(temp_path)
                self.aoi_status.value = f'<span style="color:green">✓ {filename}</span>'
            except Exception as e:
                self.aoi_status.value = f'<span style="color:red">Помилка: {e}</span>'
                self.aoi_gdf = None
    
    def _on_check_click(self, button):
        with self.status_output:
            clear_output()
            oblast = self.oblasts_gdf.loc[self.oblast_dropdown.value] if self.oblast_dropdown.value is not None else None
            print(f"📍 Область: {oblast['display_name'] if oblast is not None else 'не вибрано'}")
            
            if self.planet_key.value:
                planet = PlanetClient(self.planet_key.value)
                if planet.test_connection():
                    print(f"   Мозаїк: {len(planet.get_mosaic_list())}")
            
            if self.firms_key.value and oblast is not None:
                firms = FIRMSClient(self.firms_key.value)
                start_date = datetime.combine(self.fire_start_date.value, datetime.min.time())
                end_date = datetime.combine(self.fire_end_date.value, datetime.max.time())
                fires_gdf = firms.get_fires(oblast.geometry.bounds, start_date, end_date, source=self.firms_source.value)
                if not fires_gdf.empty:
                    fires_in = fires_gdf[fires_gdf.within(oblast.geometry)]
                    print(f"\n🔥 Точок горіння: {len(fires_in)}")
                    if not fires_in.empty:
                        print(f"\n📍 Координати:")
                        for _, row in fires_in.iterrows():
                            print(f"   {row['latitude']:.5f}, {row['longitude']:.5f} | {str(row.get('acq_date', ''))[:10]}")
    
    def _on_start_click(self, button):
        with self.status_output:
            clear_output()
            if self.oblast_dropdown.value is None:
                print('⚠️ Оберіть область')
                return
            mode = self.mode_radio.value
            if mode == 'fire_areas' and (not self.firms_key.value or not self.planet_key.value):
                print('⚠️ Введіть API ключі')
                return
            self._run_download()
    
    def _run_download(self):
        mode = self.mode_radio.value
        oblast = self.oblasts_gdf.loc[self.oblast_dropdown.value]
        self.progress.value = 0
        self.progress.layout.display = 'block'
        self.log = SessionLog(region_name=oblast['display_name'], download_mode=mode)
        
        try:
            if mode == 'fire_areas': self._download_fire_areas(oblast)
            elif mode == 'full_oblast': self._download_full_oblast(oblast)
            elif mode == 'aoi': self._download_aoi(oblast)
            print(f'\n✅ Завершено! Файли: {self.log.output_folder}')
        except Exception as e:
            self.log.add_error(str(e))
            print(f'\n❌ Помилка: {e}')
        finally:
            if self.log.output_folder:
                os.makedirs(os.path.join(self.log.output_folder, 'logs'), exist_ok=True)
                self.log.save(os.path.join(self.log.output_folder, 'logs'))
            self.progress.value = 100
    
    def _download_fire_areas(self, oblast):
        print(f'🔥 Область: {oblast["display_name"]}')
        start_date = datetime.combine(self.fire_start_date.value, datetime.min.time())
        end_date = datetime.combine(self.fire_end_date.value, datetime.max.time())
        
        comparison_config = ComparisonConfig(
            include_previous_months=self.include_prev_months.value,
            previous_months_count=self.prev_months_count.value,
            include_previous_year=self.include_prev_year.value,
            previous_years_count=self.prev_years_count.value,
            include_next_month=self.include_next_month.value
        )
        
        self.log.fire_period = (start_date, end_date)
        folders = create_folder_structure(self.output_folder.value, oblast.get('name_en', oblast['display_name']), 'fire_areas', (start_date, end_date), comparison_config)
        self.log.output_folder = folders['session']
        
        oblast_gdf = gpd.GeoDataFrame([oblast], crs="EPSG:4326")
        save_geodataframe(oblast_gdf, folders['data'], 'oblast_boundary.geojson')
        self.progress.value = 10
        
        firms_client = FIRMSClient(self.firms_key.value)
        fires_gdf = firms_client.get_fires(oblast.geometry.bounds, start_date, end_date, source=self.firms_source.value)
        if fires_gdf.empty: print('⚠️ Пожеж не знайдено'); return
        
        fires_in_oblast = fires_gdf[fires_gdf.within(oblast.geometry)]
        if fires_in_oblast.empty: print('⚠️ Пожежі за межами області'); return
        print(f'🔥 Знайдено {len(fires_in_oblast)} точок')
        save_geodataframe(fires_in_oblast, folders['data'], 'firms_fire_data.geojson')
        self.progress.value = 25
        
        clusters = FIRMSClient.cluster_fires(fires_in_oblast, buffer_km=2.0)
        if not clusters: return
        print(f'📊 Кластерів: {len(clusters)}')
        fire_areas_gdf = gpd.GeoDataFrame({'cluster_id': [c['id'] for c in clusters], 'n_hotspots': [c['n_hotspots'] for c in clusters]}, geometry=[c['geometry'] for c in clusters], crs="EPSG:4326")
        save_geodataframe(fire_areas_gdf, folders['data'], 'fire_clusters.geojson')
        self.progress.value = 30
        
        planet = PlanetClient(self.planet_key.value)
        if not planet.test_connection(): return
        
        fire_months = set()
        current = start_date
        while current <= end_date:
            fire_months.add((str(current.year), f"{current.month:02d}"))
            current += timedelta(days=28)
        fire_months.add((str(end_date.year), f"{end_date.month:02d}"))
        
        all_periods = [{'year': y, 'month': m, 'type': 'fire_period', 'description': f'Пожежі {y}-{m}', 'folder': folders.get('fire_period', folders['session'])} for y, m in sorted(fire_months)]
        
        for cp in comparison_config.generate_periods(start_date, end_date):
            cp_months = set()
            cur = cp['start']
            while cur <= cp['end']:
                cp_months.add((str(cur.year), f"{cur.month:02d}"))
                cur += timedelta(days=28)
            cp_months.add((str(cp['end'].year), f"{cp['end'].month:02d}"))
            folder_key = f"comparison_{cp['folder_suffix']}"
            for y, m in sorted(cp_months):
                all_periods.append({'year': y, 'month': m, 'type': cp['type'], 'description': f"{cp['description']} ({y}-{m})", 'folder': folders.get(folder_key, folders['session'])})
        
        print(f'📊 Періодів: {len(all_periods)}')
        total_stats = {'total': 0, 'downloaded': 0}
        
        for i, period in enumerate(all_periods, 1):
            self.progress.value = 30 + int(65 * i / len(all_periods))
            print(f'📥 [{i}/{len(all_periods)}] {period["description"]}')
            mosaic = planet.find_mosaic(period['year'], period['month'])
            if not mosaic: print('   ❌ Мозаїка не знайдена'); continue
            
            period_dir = os.path.join(period['folder'], f"{period['year']}_{period['month']}")
            os.makedirs(period_dir, exist_ok=True)
            quads = planet.get_quads_for_aoi(mosaic['id'], fire_areas_gdf)
            if not quads: continue
            
            total_stats['total'] += len(quads)
            gpd.GeoDataFrame([{"quad_id": q["id"], "geometry": box(*q["bbox"])} for q in quads]).set_crs(epsg=4326).to_file(os.path.join(period_dir, "index.geojson"), driver='GeoJSON')
            exists, downloaded, failed = planet.download_tiles(quads, period_dir)
            total_stats['downloaded'] += downloaded
            print(f'   ✅ {exists} існ., {downloaded} зав., {failed} пом.')
        
        print(f'\n📊 Всього тайлів: {total_stats["total"]}, завантажено: {total_stats["downloaded"]}')
    
    def _download_full_oblast(self, oblast):
        print(f'🛰️ Область: {oblast["display_name"]}')
        years, months = list(self.years_select.value), list(self.months_select.value)
        folders = create_folder_structure(self.output_folder.value, oblast.get('name_en', oblast['display_name']).replace(' ', '_'), 'full_oblast')
        self.log.output_folder = folders['session']
        
        oblast_gdf = gpd.GeoDataFrame([oblast], crs="EPSG:4326")
        save_geodataframe(oblast_gdf, folders['data'], 'oblast_boundary.geojson')
        
        planet = PlanetClient(self.planet_key.value)
        if not planet.test_connection(): raise Exception("Planet API недоступний")
        
        available = [(y, m, planet.find_mosaic(y, m)) for y in years for m in months if planet.find_mosaic(y, m)]
        for i, (year, month, mosaic) in enumerate(available):
            self.progress.value = int(100 * (i + 1) / len(available))
            print(f'📥 [{i+1}/{len(available)}] {year}-{month}')
            period_dir = os.path.join(folders['session'], f"{year}_{month}")
            os.makedirs(period_dir, exist_ok=True)
            quads = planet.get_quads_for_aoi(mosaic['id'], oblast_gdf)
            if quads:
                gpd.GeoDataFrame([{"quad_id": q["id"], "geometry": box(*q["bbox"])} for q in quads]).set_crs(epsg=4326).to_file(os.path.join(period_dir, "index.geojson"), driver='GeoJSON')
                exists, downloaded, failed = planet.download_tiles(quads, period_dir)
                print(f'   ✅ {exists} існ., {downloaded} зав., {failed} пом.')
    
    def _download_aoi(self, oblast):
        print(f'📍 Зона інтересу')
        if self.aoi_gdf is None: print('⚠️ Завантажте файл'); return
        folders = create_folder_structure(self.output_folder.value, 'AOI', 'aoi')
        self.log.output_folder = folders['session']
        save_geodataframe(self.aoi_gdf, folders['data'], 'aoi_boundary.geojson')
        
        if self.planet_key.value:
            planet = PlanetClient(self.planet_key.value)
            if planet.test_connection():
                years = list(self.years_select.value) or [str(datetime.now().year)]
                months = list(self.months_select.value) or [f"{datetime.now().month:02d}"]
                for y in years:
                    for m in months:
                        mosaic = planet.find_mosaic(y, m)
                        if mosaic:
                            period_dir = os.path.join(folders['session'], f"{y}_{m}")
                            os.makedirs(period_dir, exist_ok=True)
                            quads = planet.get_quads_for_aoi(mosaic['id'], self.aoi_gdf)
                            if quads: planet.download_tiles(quads, period_dir)
    
    def display(self):
        ui = widgets.VBox([
            widgets.HTML('<h2>Forest Fire Risk RS</h2>'),
            self.oblast_dropdown, self.mode_radio,
            self.fire_options_box, self.aoi_options_box, self.full_oblast_options_box,
            widgets.HTML('<b>API ключі:</b>'), self.planet_key, self.firms_key,
            widgets.HTML('<b>Вивід:</b>'), self.output_folder,
            widgets.HBox([self.check_btn, self.start_button]),
            self.progress, self.status_output
        ])
        display(ui)
        self._on_mode_change({'new': self.mode_radio.value})

In [ ]:
ui = ForestFireUI()
ui.display()